# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb pandas pyarrow huggingface_hub scikit-learn

import duckdb, getpass
from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_TOKEN")      # Colab Secrets (key icon), name: HF_TOKEN
except Exception:
    HF_TOKEN = getpass.getpass("HF read token: ")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
D = f"{REL}/fact_content_daily_performance/**/*.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"

In [ ]:
# 1. Confirm source tables and schema

print("Daily fact source:", D)
print("Content dimension:", DIM_CONTENT)

print("\nDaily fact columns:")
display(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{D}', hive_partitioning=true)").df())

print("\nContent dimension columns:")
display(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DIM_CONTENT}')").df())


Daily fact source: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet
Content dimension: hf://datasets/FlyRank/internship-warehouse/dim_content.parquet

Daily fact columns:


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



Content dimension columns:


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [ ]:
print(con.sql(f"""
SELECT COUNT(*) AS n_content,
       AVG(is_published::INT) AS pub_rate,
       AVG(is_deleted::INT)   AS deleted_rate,
       COUNT(DISTINCT content_type) AS n_types,
       COUNT(DISTINCT main_intent)  AS n_intents
FROM read_parquet('{DIM_CONTENT}')
""").df().to_string())

D = D

client_diag = con.sql(f"""
SELECT client_hash_id,
       MIN(report_date) AS first_row,
       MIN(report_date) FILTER (WHERE gsc_data_available) AS first_gsc,
       MAX(report_date) FILTER (WHERE gsc_data_available) AS last_gsc,
       COUNT(*) AS n_rows,
       SUM(gsc_data_available::INT) AS gsc_rows,
       COUNT(DISTINCT content_hash_id) AS contents,
       COUNT(DISTINCT content_hash_id) FILTER (WHERE gsc_data_available) AS contents_with_gsc,
       SUM(gsc_clicks) AS clicks
FROM read_parquet('{D}', hive_partitioning=true)
GROUP BY 1
""").df()
client_diag["gsc_row_share"] = client_diag.gsc_rows / client_diag.n_rows
client_diag["content_with_gsc_share"] = client_diag.contents_with_gsc / client_diag.contents
client_diag["click_share"] = client_diag.clicks / client_diag.clicks.sum()
print(client_diag[["gsc_row_share","content_with_gsc_share","click_share"]].describe().to_string())
print("top-5 clients' share of clicks:", client_diag.click_share.nlargest(5).sum().round(3))
print((client_diag.first_gsc.dt.to_period("M")).value_counts().sort_index())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_content  pub_rate  deleted_rate  n_types  n_intents
0     519606  0.792023      0.195454        3          4


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       gsc_row_share  content_with_gsc_share  click_share
count      70.000000               70.000000    70.000000
mean        0.285459                0.648270     0.014286
std         0.299939                0.347097     0.040204
min         0.000000                0.000000     0.000000
25%         0.037096                0.362683     0.000013
50%         0.153975                0.768954     0.000265
75%         0.481831                0.955415     0.004412
max         0.903980                1.000000     0.262519
top-5 clients' share of clicks: 0.673
first_gsc
2025-01     2
2025-02     1
2025-03     1
2025-06     5
2025-07     7
2025-09     8
2025-10     5
2025-11    10
2025-12     1
2026-01     1
2026-02    11
2026-03     5
2026-04     5
2026-05     4
2026-06     1
Freq: M, Name: count, dtype: int64


In [ ]:
print(con.sql(f"""
SELECT COUNT(*) AS n_content,
       AVG(is_published::INT) AS pub_rate,
       AVG(is_deleted::INT)   AS deleted_rate,
       COUNT(DISTINCT content_type) AS n_types,
       COUNT(DISTINCT main_intent)  AS n_intents
FROM read_parquet('{DIM_CONTENT}')
""").df().to_string())

D = D

client_diag = con.sql(f"""
SELECT client_hash_id,
       MIN(report_date) AS first_row,
       MIN(report_date) FILTER (WHERE gsc_data_available) AS first_gsc,
       MAX(report_date) FILTER (WHERE gsc_data_available) AS last_gsc,
       COUNT(*) AS n_rows,
       SUM(gsc_data_available::INT) AS gsc_rows,
       COUNT(DISTINCT content_hash_id) AS contents,
       COUNT(DISTINCT content_hash_id) FILTER (WHERE gsc_data_available) AS contents_with_gsc,
       SUM(gsc_clicks) AS clicks
FROM read_parquet('{D}', hive_partitioning=true)
GROUP BY 1
""").df()
client_diag["gsc_row_share"] = client_diag.gsc_rows / client_diag.n_rows
client_diag["content_with_gsc_share"] = client_diag.contents_with_gsc / client_diag.contents
client_diag["click_share"] = client_diag.clicks / client_diag.clicks.sum()
print(client_diag[["gsc_row_share","content_with_gsc_share","click_share"]].describe().to_string())
print("top-5 clients' share of clicks:", client_diag.click_share.nlargest(5).sum().round(3))
print((client_diag.first_gsc.dt.to_period("M")).value_counts().sort_index())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_content  pub_rate  deleted_rate  n_types  n_intents
0     519606  0.792023      0.195454        3          4


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       gsc_row_share  content_with_gsc_share  click_share
count      70.000000               70.000000    70.000000
mean        0.285459                0.648270     0.014286
std         0.299939                0.347097     0.040204
min         0.000000                0.000000     0.000000
25%         0.037096                0.362683     0.000013
50%         0.153975                0.768954     0.000265
75%         0.481831                0.955415     0.004412
max         0.903980                1.000000     0.262519
top-5 clients' share of clicks: 0.673
first_gsc
2025-01     2
2025-02     1
2025-03     1
2025-06     5
2025-07     7
2025-09     8
2025-10     5
2025-11    10
2025-12     1
2026-01     1
2026-02    11
2026-03     5
2026-04     5
2026-05     4
2026-06     1
Freq: M, Name: count, dtype: int64


In [ ]:
import os
os.makedirs("/content/cache", exist_ok=True)
con.execute(f"""
COPY (
  SELECT report_date, client_hash_id, content_hash_id,
         gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position
  FROM read_parquet('{D}', hive_partitioning=true)
  WHERE gsc_data_available
    AND report_date >= DATE '2025-09-01'
    AND report_date < DATE '2026-07-01'
) TO '/content/cache/daily_gsc.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print(round(os.path.getsize("/content/cache/daily_gsc.parquet")/1e9, 2), "GB")
print(con.sql("SELECT COUNT(*) n, COUNT(DISTINCT content_hash_id) c FROM '/content/cache/daily_gsc.parquet'").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

0.36 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

          n       c
0  26585916  307186


In [ ]:
import pandas as pd, numpy as np

CACHE = "/content/cache/daily_gsc.parquet"
con.execute(f"CREATE OR REPLACE VIEW daily AS SELECT * FROM read_parquet('{CACHE}')")
con.execute(f"CREATE OR REPLACE VIEW dimc  AS SELECT * FROM read_parquet('{REL}/dim_content.parquet')")
print(con.sql("SELECT MIN(report_date) mn, MAX(report_date) mx FROM daily").df())

CUTOFFS = ["2026-01-01","2026-02-01","2026-03-01","2026-04-01","2026-05-01","2026-06-01"]

def build_panel(T):
    T = pd.Timestamp(T)
    d = lambda k: (T + pd.Timedelta(days=k)).strftime("%Y-%m-%d")
    W = lambda a, b: f"report_date >= DATE '{d(a)}' AND report_date < DATE '{d(b)}'"
    sql = f"""
    WITH client_ok AS (
      SELECT client_hash_id FROM daily
      GROUP BY client_hash_id
      HAVING MIN(report_date) <= DATE '{d(-60)}'
         AND COUNT(*) FILTER (WHERE {W(-7,0)})  > 0
         AND COUNT(*) FILTER (WHERE {W(23,30)}) > 0
    ),
    agg AS (
      SELECT client_hash_id, content_hash_id,
        SUM(gsc_clicks) FILTER (WHERE {W(-7,0)})    AS c7,
        SUM(gsc_clicks) FILTER (WHERE {W(-30,0)})   AS c30,
        SUM(gsc_clicks) FILTER (WHERE {W(-60,-30)}) AS cp30,
        SUM(gsc_clicks) FILTER (WHERE {W(-90,0)})   AS c90,
        SUM(gsc_impressions) FILTER (WHERE {W(-30,0)})   AS i30,
        SUM(gsc_impressions) FILTER (WHERE {W(-60,-30)}) AS ip30,
        SUM(gsc_impressions) FILTER (WHERE {W(-90,0)})   AS i90,
        SUM(gsc_avg_position*gsc_impressions) FILTER (WHERE {W(-30,0)} AND gsc_avg_position IS NOT NULL)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE {W(-30,0)} AND gsc_avg_position IS NOT NULL),0) AS pos30,
        SUM(gsc_avg_position*gsc_impressions) FILTER (WHERE {W(-60,-30)} AND gsc_avg_position IS NOT NULL)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE {W(-60,-30)} AND gsc_avg_position IS NOT NULL),0) AS pos_p30,
        COUNT(*) FILTER (WHERE {W(-30,0)})   AS act30,
        COUNT(*) FILTER (WHERE {W(-60,-30)}) AS actp30,
        COUNT(*) FILTER (WHERE {W(-90,0)})   AS act90,
        MAX(gsc_clicks) FILTER (WHERE {W(-30,0)}) AS maxday30,
        STDDEV_SAMP(gsc_clicks) FILTER (WHERE {W(-90,0)}) AS sd90,
        COALESCE(SUM(gsc_clicks) FILTER (WHERE {W(0,30)}),0)      AS y_clicks,
        COALESCE(SUM(gsc_impressions) FILTER (WHERE {W(0,30)}),0) AS y_impr
      FROM daily
      WHERE report_date >= DATE '{d(-90)}' AND report_date < DATE '{d(30)}'
        AND client_hash_id IN (SELECT client_hash_id FROM client_ok)
      GROUP BY 1,2
    )
    SELECT DATE '{d(0)}' AS cutoff, a.*,
      c.content_type, c.main_intent, c.word_count, c.search_volume, c.backlinks,
      c.competition, c.keyword_token_count,
      date_diff('day', c.content_created_date, DATE '{d(0)}') AS age_days
    FROM agg a JOIN dimc c USING (client_hash_id, content_hash_id)
    WHERE COALESCE(c.is_deleted,false) = false AND COALESCE(c.is_published,true) = true
      AND COALESCE(a.c30,0) >= 10 AND COALESCE(a.i30,0) >= 100   -- uses feature window only
    """
    return con.sql(sql).df()

panels = []
for T in CUTOFFS:
    p = build_panel(T); panels.append(p)
    print(T, len(p), "rows,", p.client_hash_id.nunique(), "clients")
panel = pd.concat(panels, ignore_index=True)

# label variants (use future window; never used as features)
g = panel.groupby(["cutoff","client_hash_id"])
panel["client_ratio"] = g.y_clicks.transform("sum") / g.c30.transform("sum")
panel["ratio"] = panel.y_clicks / panel.c30
panel["y_raw"] = (panel.ratio <= 0.75).astype(int)                       # absolute drop >=25%
panel["y_rel"] = ((panel.ratio / panel.client_ratio.replace(0,np.nan)) <= 0.75).astype(int)  # drop vs own client trend

print(panel.groupby("cutoff")[["y_raw","y_rel"]].mean().round(3))
print(panel.groupby("cutoff").client_hash_id.nunique())
share = panel.groupby(["cutoff","client_hash_id"]).size().groupby("cutoff").apply(lambda s: s.max()/s.sum())
print("largest client's share of rows per cutoff:\n", share.round(3))
panel.to_parquet("/content/cache/panel.parquet")

          mn         mx
0 2025-09-01 2026-06-30


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-01-01 6153 rows, 16 clients


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-02-01 9054 rows, 18 clients


...

2026-04-01 14495 rows, 21 clients


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-05-01 16286 rows, 30 clients


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-06-01 17911 rows, 31 clients
            y_raw  y_rel
cutoff                  
2026-01-01  0.263  0.350
2026-02-01  0.249  0.290
2026-03-01  0.337  0.324
2026-04-01  0.481  0.389
2026-05-01  0.410  0.353
2026-06-01  0.615  0.370
cutoff
2026-01-01    16
2026-02-01    18
2026-03-01    19
2026-04-01    21
2026-05-01    30
2026-06-01    31
Name: client_hash_id, dtype: int64
largest client's share of rows per cutoff:
 cutoff
2026-01-01    0.316
2026-02-01    0.281
2026-03-01    0.289
2026-04-01    0.302
2026-05-01    0.265
2026-06-01    0.258
dtype: float64


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold

P = pd.read_parquet("/content/cache/panel.parquet").copy()
P["cutoff"] = pd.to_datetime(P["cutoff"])
LABEL = "y_rel"

# ---- features: feature-window columns only (no y_*, no client_ratio, no ratio) ----
eps = 1.0
P["lc30"], P["lc90"] = np.log1p(P.c30), np.log1p(P.c90)
P["lcpm"]   = np.log1p(P.cp30)
P["mom"]    = np.log((P.c30+eps)/(P.cp30+eps))                       # click momentum
P["spike"]  = np.log((P.c30+eps)/((P.c90/3)+eps))                    # last 30 vs 90d avg
P["recent"] = np.log((P.c7*30/7+eps)/(P.c30+eps))                    # last week vs month
P["ctr30"]  = P.c30/(P.i30+eps)
P["ctr_chg"]= P.ctr30 - P.cp30.div(P.ip30.replace(0,np.nan))
P["impr_mom"]= np.log((P.i30+eps)/(P.ip30+eps))
P["pos_chg"]= P.pos30 - P.pos_p30
P["act_frac30"] = P.act30/30
P["cv90"]   = P.sd90/((P.c90/90)+eps)
P["peak_share"] = P.maxday30/(P.c30+eps)
# client context from the feature window only (known at cutoff)
cg = P.groupby(["cutoff","client_hash_id"])
P["client_mom"] = np.log((cg.c30.transform("sum")+eps)/(cg.cp30.transform("sum")+eps))
P["mom_vs_client"] = P.mom - P.client_mom
P["share_of_client"] = P.c30/(cg.c30.transform("sum")+eps)
P["lcl_pages"] = np.log1p(cg.c30.transform("size"))
for c in ["content_type","main_intent"]:
    P[c+"_code"] = P[c].astype("category").cat.codes

NUM = ["lc30","lc90","lcpm","mom","spike","recent","ctr30","ctr_chg","impr_mom","pos30","pos_chg",
       "act_frac30","cv90","peak_share","client_mom","mom_vs_client","share_of_client","lcl_pages",
       "word_count","search_volume","backlinks","competition","keyword_token_count","age_days"]
CAT = ["content_type_code","main_intent_code"]
FEATS = NUM + CAT
assert not any(f.startswith("y_") or f in ("ratio","client_ratio") for f in FEATS)

# ---- baselines: higher score = more likely to decline ----
P["base_mom"]  = -P.mom_vs_client.fillna(0)         # simple momentum, client-adjusted
P["base_spike"]= P.spike.fillna(0)                  # coming off a spike -> mean reversion

# ---- splits ----
tr = P.cutoff <= "2026-03-01"; va = P.cutoff == "2026-04-01"; te = P.cutoff >= "2026-05-01"
w = 1/np.sqrt(P.groupby(["cutoff","client_hash_id"]).c30.transform("size"))   # damp big clients

def fit_models(Xtr, ytr, wtr):
    lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LogisticRegression(max_iter=2000, C=0.3))
    lr.fit(Xtr[NUM], ytr, logisticregression__sample_weight=wtr)
    gb = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300,
            l2_regularization=1.0, min_samples_leaf=50, categorical_features=[FEATS.index(c) for c in CAT],
            random_state=0)
    gb.fit(Xtr[FEATS], ytr, sample_weight=wtr)
    return lr, gb

def score_all(lr, gb, D):
    return {"baseline_momentum": D.base_mom.values, "baseline_spike": D.base_spike.values,
            "logistic": lr.predict_proba(D[NUM])[:,1], "hist_gb": gb.predict_proba(D[FEATS])[:,1]}

def p_at_k(y, s, k=0.10):
    n = max(1,int(len(y)*k)); idx = np.argsort(-s)[:n]; return y[idx].mean()

def report(D, scores, label=LABEL):
    y = D[label].values; rows = []
    for name, s in scores.items():
        per_cut = [p_at_k(y[m], s[m]) for c in D.cutoff.unique() for m in [(D.cutoff==c).values]]
        rows.append(dict(model=name, pr_auc=average_precision_score(y,s), roc_auc=roc_auc_score(y,s),
                         p_at_10pct=np.mean(per_cut), base_rate=y.mean()))
    return pd.DataFrame(rows).round(3)

# ---- A) time-forward split ----
lr, gb = fit_models(P[tr], P.loc[tr,LABEL], w[tr])
print("VALIDATION (Apr)");  print(report(P[va], score_all(lr,gb,P[va])).to_string(index=False))
# refit on train+val for the final test (hyperparameters were not tuned on test)
trva = tr | va
lr, gb = fit_models(P[trva], P.loc[trva,LABEL], w[trva])
S_te = score_all(lr, gb, P[te])
print("\nTEST (May+Jun)");   print(report(P[te], S_te).to_string(index=False))

# ---- B) client-bootstrap CI: model minus baseline PR-AUC on test ----
D = P[te].reset_index(drop=True); y = D[LABEL].values; cl = D.client_hash_id.values
rng = np.random.default_rng(0); uc = np.unique(cl); diffs = []
idx_by_c = {c: np.where(cl==c)[0] for c in uc}
for _ in range(200):
    pick = np.concatenate([idx_by_c[c] for c in rng.choice(uc, len(uc))])
    if y[pick].min()==y[pick].max(): continue
    diffs.append(average_precision_score(y[pick], S_te["hist_gb"][pick]) -
                 average_precision_score(y[pick], S_te["baseline_momentum"][pick]))
print("\nPR-AUC gain, hist_gb - baseline_momentum: mean %.3f, 95%% CI [%.3f, %.3f]" %
      (np.mean(diffs), *np.percentile(diffs,[2.5,97.5])))

# ---- C) held-out-CLIENT, time-forward check ----
gkf = GroupKFold(n_splits=5); out = []
cl_all = P.client_hash_id.values
for k,(a,b) in enumerate(gkf.split(P, groups=cl_all)):
    A = P.iloc[a]; B = P.iloc[b]
    A = A[A.cutoff <= "2026-03-01"]; B = B[B.cutoff >= "2026-05-01"]
    if B[LABEL].nunique()<2 or len(A)<500: continue
    _, gbk = fit_models(A, A[LABEL], w.loc[A.index])
    s = gbk.predict_proba(B[FEATS])[:,1]
    out.append(dict(fold=k, n=len(B), pr_auc_gb=average_precision_score(B[LABEL], s),
                    pr_auc_base=average_precision_score(B[LABEL], B.base_mom), base_rate=B[LABEL].mean()))
print("\nHELD-OUT CLIENTS (train<=Mar, test>=May)"); print(pd.DataFrame(out).round(3).to_string(index=False))

# ---- D) per-client macro metric on test ----
pc = []
for c, g_ in D.groupby("client_hash_id"):
    if len(g_)>=100 and g_[LABEL].nunique()==2:
        m = g_.index
        pc.append((average_precision_score(y[m], S_te["hist_gb"][m]) - average_precision_score(y[m], S_te["baseline_momentum"][m])))
print("\nclients with enough data:", len(pc), "| share where model beats baseline:", np.mean(np.array(pc)>0).round(2),
      "| median PR-AUC gain:", np.median(pc).round(3))

VALIDATION (Apr)
            model  pr_auc  roc_auc  p_at_10pct  base_rate
baseline_momentum   0.365    0.455       0.352      0.389
   baseline_spike   0.408    0.540       0.392      0.389
         logistic   0.535    0.631       0.648      0.389
          hist_gb   0.551    0.644       0.669      0.389

TEST (May+Jun)
            model  pr_auc  roc_auc  p_at_10pct  base_rate
baseline_momentum   0.387    0.494       0.494      0.362
   baseline_spike   0.374    0.514       0.321      0.362
         logistic   0.502    0.634       0.610      0.362
          hist_gb   0.490    0.622       0.580      0.362

PR-AUC gain, hist_gb - baseline_momentum: mean 0.102, 95% CI [0.053, 0.146]

HELD-OUT CLIENTS (train<=Mar, test>=May)
 fold    n  pr_auc_gb  pr_auc_base  base_rate
    0 8936      0.446        0.292      0.314
    1 6155      0.428        0.356      0.329
    2 3969      0.440        0.309      0.327
    3 8531      0.447        0.400      0.368
    4 6606      0.635        0.560    

In [ ]:
from sklearn.inspection import permutation_importance

# Use the same leakage-safe feature list defined in the modelling section.
FEATS = NUM + CAT

# fair baselines: both directions
P["base_mom_flip"] = P.mom_vs_client.fillna(0)   # recent growth -> predicted decline
lr, gb = fit_models(P[trva], P.loc[trva, LABEL], w[trva])
S = score_all(lr, gb, P[te]); S["baseline_momentum_flipped"] = P.loc[te, "base_mom_flip"].values

print("TEST (May+Jun), with sign-flipped baseline"); print(report(P[te], S).to_string(index=False))
for c in sorted(P.loc[te, "cutoff"].unique()):
    m = (P[te].cutoff == c).values
    print(str(c)[:10], report(P[te][m], {k: v[m] for k, v in S.items()}).set_index("model")[["pr_auc","p_at_10pct","base_rate"]].round(3).to_dict("index"))

# gain vs the BEST baseline direction, client bootstrap
D = P[te].reset_index(drop=True); y = D[LABEL].values; cl = D.client_hash_id.values
idx_by_c = {c: np.where(cl == c)[0] for c in np.unique(cl)}; rng = np.random.default_rng(1); d2 = []
for _ in range(200):
    pick = np.concatenate([idx_by_c[c] for c in rng.choice(list(idx_by_c), len(idx_by_c))])
    if y[pick].min() == y[pick].max(): continue
    d2.append(average_precision_score(y[pick], S["hist_gb"][pick]) -
              average_precision_score(y[pick], S["baseline_momentum_flipped"][pick]))
print("\nPR-AUC gain vs FLIPPED baseline: mean %.3f, 95%% CI [%.3f, %.3f]" % (np.mean(d2), *np.percentile(d2, [2.5, 97.5])))

# permutation importance on test (PR-AUC drop)
pi = permutation_importance(gb, P.loc[te, FEATS], P.loc[te, LABEL], scoring="average_precision",
                            n_repeats=5, random_state=0)
print("\nTop features by permutation importance:")
print(pd.Series(pi.importances_mean, index=FEATS).sort_values(ascending=False).head(12).round(4).to_string())

# ablations: which feature groups carry the signal?
def gb_only(drop):
    F = [f for f in FEATS if f not in drop]
    g = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, l2_regularization=1.0,
            min_samples_leaf=50, categorical_features=[F.index(c) for c in CAT if c in F], random_state=0)
    g.fit(P.loc[trva, F], P.loc[trva, LABEL], sample_weight=w[trva])
    return average_precision_score(P.loc[te, LABEL], g.predict_proba(P.loc[te, F])[:, 1])

groups = {"all features": [],
          "no client-context": ["client_mom","mom_vs_client","share_of_client","lcl_pages"],
          "no momentum/spike": ["mom","spike","recent","impr_mom","mom_vs_client"],
          "no content metadata": ["word_count","search_volume","backlinks","competition","keyword_token_count","age_days","content_type_code","main_intent_code"]}
print("\nAblation (test PR-AUC):"); [print(f"  {k:22s} {gb_only(v):.3f}") for k, v in groups.items()]

# lift by score decile on test
T_ = P[te].copy(); T_["score"] = S["hist_gb"]
T_["decile"] = pd.qcut(T_.score.rank(method="first"), 10, labels=False) + 1
print("\nDecile (10 = highest risk):"); print(T_.groupby("decile").agg(mean_score=("score","mean"), actual_rate=(LABEL,"mean"), n=(LABEL,"size")).round(3).to_string())

TEST (May+Jun), with sign-flipped baseline
                    model  pr_auc  roc_auc  p_at_10pct  base_rate
        baseline_momentum   0.387    0.494       0.494      0.362
           baseline_spike   0.374    0.514       0.321      0.362
                 logistic   0.502    0.634       0.610      0.362
                  hist_gb   0.490    0.622       0.580      0.362
baseline_momentum_flipped   0.374    0.506       0.386      0.362
2026-05-01 {'baseline_momentum': {'pr_auc': 0.329, 'p_at_10pct': 0.324, 'base_rate': 0.353}, 'baseline_spike': {'pr_auc': 0.401, 'p_at_10pct': 0.289, 'base_rate': 0.353}, 'logistic': {'pr_auc': 0.478, 'p_at_10pct': 0.533, 'base_rate': 0.353}, 'hist_gb': {'pr_auc': 0.468, 'p_at_10pct': 0.542, 'base_rate': 0.353}, 'baseline_momentum_flipped': {'pr_auc': 0.396, 'p_at_10pct': 0.388, 'base_rate': 0.353}}
2026-06-01 {'baseline_momentum': {'pr_auc': 0.459, 'p_at_10pct': 0.663, 'base_rate': 0.37}, 'baseline_spike': {'pr_auc': 0.355, 'p_at_10pct': 0.352, 'base_rat

In [ ]:

# 1) scoring panel (no label window exists, so skip the label-window client check)
def build_panel_score(T):
    T = pd.Timestamp(T); d = lambda k: (T + pd.Timedelta(days=k)).strftime("%Y-%m-%d")
    W = lambda a, b: f"report_date >= DATE '{d(a)}' AND report_date < DATE '{d(b)}'"
    sql = f"""
    WITH client_ok AS (
      SELECT client_hash_id FROM daily GROUP BY 1
      HAVING MIN(report_date) <= DATE '{d(-60)}' AND COUNT(*) FILTER (WHERE {W(-7,0)}) > 0),
    agg AS (
      SELECT client_hash_id, content_hash_id,
        SUM(gsc_clicks) FILTER (WHERE {W(-7,0)})    AS c7,
        SUM(gsc_clicks) FILTER (WHERE {W(-30,0)})   AS c30,
        SUM(gsc_clicks) FILTER (WHERE {W(-60,-30)}) AS cp30,
        SUM(gsc_clicks) FILTER (WHERE {W(-90,0)})   AS c90,
        SUM(gsc_impressions) FILTER (WHERE {W(-30,0)})   AS i30,
        SUM(gsc_impressions) FILTER (WHERE {W(-60,-30)}) AS ip30,
        SUM(gsc_impressions) FILTER (WHERE {W(-90,0)})   AS i90,
        SUM(gsc_avg_position*gsc_impressions) FILTER (WHERE {W(-30,0)})
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE {W(-30,0)}),0)   AS pos30,
        SUM(gsc_avg_position*gsc_impressions) FILTER (WHERE {W(-60,-30)})
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE {W(-60,-30)}),0) AS pos_p30,
        COUNT(*) FILTER (WHERE {W(-30,0)})   AS act30,
        COUNT(*) FILTER (WHERE {W(-60,-30)}) AS actp30,
        COUNT(*) FILTER (WHERE {W(-90,0)})   AS act90,
        MAX(gsc_clicks) FILTER (WHERE {W(-30,0)}) AS maxday30,
        STDDEV_SAMP(gsc_clicks) FILTER (WHERE {W(-90,0)}) AS sd90
      FROM daily WHERE report_date >= DATE '{d(-90)}' AND report_date < DATE '{d(0)}'
        AND client_hash_id IN (SELECT client_hash_id FROM client_ok)
      GROUP BY 1,2)
    SELECT DATE '{d(0)}' AS cutoff, a.*, c.content_type, c.main_intent, c.word_count, c.search_volume,
      c.backlinks, c.competition, c.keyword_token_count,
      date_diff('day', c.content_created_date, DATE '{d(0)}') AS age_days
    FROM agg a JOIN dimc c USING (client_hash_id, content_hash_id)
    WHERE COALESCE(c.is_deleted,false)=false AND COALESCE(c.is_published,true)=true
      AND COALESCE(a.c30,0) >= 10 AND COALESCE(a.i30,0) >= 100"""
    return con.sql(sql).df()

S_ = build_panel_score("2026-07-01"); eps = 1.0
S_["lc30"], S_["lc90"], S_["lcpm"] = np.log1p(S_.c30), np.log1p(S_.c90), np.log1p(S_.cp30)
S_["mom"] = np.log((S_.c30+eps)/(S_.cp30+eps)); S_["spike"] = np.log((S_.c30+eps)/((S_.c90/3)+eps))
S_["recent"] = np.log((S_.c7*30/7+eps)/(S_.c30+eps)); S_["ctr30"] = S_.c30/(S_.i30+eps)
S_["ctr_chg"] = S_.ctr30 - S_.cp30.div(S_.ip30.replace(0,np.nan)); S_["impr_mom"] = np.log((S_.i30+eps)/(S_.ip30+eps))
S_["pos_chg"] = S_.pos30 - S_.pos_p30; S_["act_frac30"] = S_.act30/30
S_["cv90"] = S_.sd90/((S_.c90/90)+eps); S_["peak_share"] = S_.maxday30/(S_.c30+eps)
cg = S_.groupby("client_hash_id")
S_["client_mom"] = np.log((cg.c30.transform("sum")+eps)/(cg.cp30.transform("sum")+eps))
S_["mom_vs_client"] = S_.mom - S_.client_mom; S_["share_of_client"] = S_.c30/(cg.c30.transform("sum")+eps)
S_["lcl_pages"] = np.log1p(cg.c30.transform("size"))
print("scoring set:", len(S_), "pages,", S_.client_hash_id.nunique(), "clients")

# 2) lean-feature check (is the simpler model as good on the test split?)
LEAN = ["recent","cv90","spike","impr_mom","peak_share","ctr_chg","mom","lc30","ctr30","pos30","pos_chg","act_frac30"]
def mk(F): return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000, C=0.3))
for name, F in [("full logistic", NUM), ("lean logistic", LEAN)]:
    m = mk(F); m.fit(P.loc[trva, F], P.loc[trva, LABEL], logisticregression__sample_weight=w[trva])
    print(name, "test PR-AUC", round(average_precision_score(P.loc[te, LABEL], m.predict_proba(P.loc[te, F])[:,1]), 3))

# 3) final model on all labeled cutoffs (lean features; labels all end by 2026-06-30)
F = LEAN; final = mk(F); final.fit(P[F], P[LABEL], logisticregression__sample_weight=w)
S_["risk"] = final.predict_proba(S_[F])[:,1]

# 4) reason codes = top positive contributions (coef x standardized value)
imp, sc, lr_ = final.steps[0][1], final.steps[1][1], final.steps[2][1]
Z = sc.transform(imp.transform(S_[F])); C = Z * lr_.coef_[0]
nice = {"recent":"last-week click pace","cv90":"click volatility","spike":"30d vs 90d click level",
        "impr_mom":"impression trend","peak_share":"single-day click concentration","ctr_chg":"CTR change",
        "mom":"30d click trend","lc30":"click volume","ctr30":"CTR level","pos30":"avg position",
        "pos_chg":"position change","act_frac30":"days with visibility"}
def reasons(i):
    top = np.argsort(-C[i])[:2]
    return "; ".join(f"{nice[F[j]]} ({'high' if Z[i,j]>0 else 'low'})" for j in top if C[i,j] > 0.05)
S_["reasons"] = [reasons(i) for i in range(len(S_))]

# 5) action buckets (decision-support; not a causal claim)
S_["risk_pct"] = S_.risk.rank(pct=True)
vol_hi = S_.groupby("client_hash_id").c30.transform(lambda s: s.rank(pct=True)) >= 0.5
def action(r, v):
    if r >= 0.90 and v: return "1-Review now (high risk, meaningful traffic)"
    if r >= 0.90:       return "2-Watch closely (high risk, smaller traffic)"
    if r >= 0.70:       return "3-Monitor"
    if r <= 0.30 and v: return "4-Protect (low risk, meaningful traffic)"
    return "5-No action"
S_["action"] = [action(r, v) for r, v in zip(S_.risk_pct, vol_hi)]
print(S_.action.value_counts().sort_index())
print(S_.groupby("action")[["risk","c30"]].agg(["mean","median"]).round(3))

# private ranked file (contains hashes): do NOT commit it to the public repo
out = S_.sort_values("risk", ascending=False)[["client_hash_id","content_hash_id","risk","action","reasons","c30","mom","recent","cv90"]]
out.to_csv("/content/cache/ranked_actions_PRIVATE.csv", index=False)
pub = out.head(15).copy()
pub["client"] = "Client " + pd.factorize(pub.client_hash_id)[0].astype(str)
pub["page"] = ["Page " + str(i+1) for i in range(len(pub))]
print(pub[["page","client","risk","action","reasons"]].round(3).to_string(index=False))

# Public-safety check: the private ranked file stays in /content/cache and must not be committed.
assert os.path.exists("/content/cache/ranked_actions_PRIVATE.csv")
assert "client_hash_id" in out.columns and "content_hash_id" in out.columns
print("Private ranked output exists locally; do not publish it or commit it.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

scoring set: 17413 pages, 33 clients
full logistic test PR-AUC 0.502
lean logistic test PR-AUC 0.506
action
1-Review now (high risk, meaningful traffic)     600
2-Watch closely (high risk, smaller traffic)    1142
3-Monitor                                       3482
4-Protect (low risk, meaningful traffic)        2798
5-No action                                     9391
Name: count, dtype: int64
                                               risk            c30       
                                               mean median    mean median
action                                                                   
1-Review now (high risk, meaningful traffic)  0.596  0.557  48.937   28.0
2-Watch closely (high risk, smaller traffic)  0.610  0.614  12.907   12.0
3-Monitor                                     0.429  0.424  30.840   18.0
4-Protect (low risk, meaningful traffic)      0.273  0.278  68.889   39.0
5-No action                                   0.322  0.327  30.354   17.0
   page  

In [ ]:
# Final self-check

checks = {
    "panel cache exists": os.path.exists("/content/cache/panel.parquet"),
    "private ranked output exists": os.path.exists("/content/cache/ranked_actions_PRIVATE.csv"),
    "evaluation includes PR-AUC": "pr_auc" in report(P[te], S).columns,
    "scoring set has rows": len(S_) > 0,
    "ranked output has risk": "risk" in out.columns,
    "ranked output has action": "action" in out.columns,
    "ranked output has reason codes": "reasons" in out.columns,
}
for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", name)
assert all(checks.values()), "One or more self-checks failed."

print("\nSubmission reminders:")
print("- Update paper metrics/charts from this successful run.")
print("- Do not publish hashed identifiers or the private ranked CSV.")
print("- Ensure submission/paper_url.txt contains only the deployed paper URL.")
print("- Commit the notebook and public paper assets, not cache/export files.")


PASS - panel cache exists
PASS - private ranked output exists
PASS - evaluation includes PR-AUC
PASS - scoring set has rows
PASS - ranked output has risk
PASS - ranked output has action
PASS - ranked output has reason codes

Submission reminders:
- Update paper metrics/charts from this successful run.
- Do not publish hashed identifiers or the private ranked CSV.
- Ensure submission/paper_url.txt contains only the deployed paper URL.
- Commit the notebook and public paper assets, not cache/export files.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
